A toy example with random data. You can play with this to have a basic understanding.

- License-Identifier: GPL-3.0-only

- This file is part of the TT-sandbox project.

- Copyright © 2025 Idiap Research Institute <contact@idiap.ch>

- Contributor: Teng Xue <teng.xue@idiap.ch>

In [1]:
import open3d as o3d 
import numpy as np
from utils import tt_svd_full, tt_svd_rank, tt_svd_thres, tt_svd_recon
from utils import svd_full, svd_rank, svd_thres, svd_recon

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# generate a tensor
shape = (3, 4, 5, 2) 
tensor = np.random.rand(*shape)

In [3]:
tt_cores_1 = tt_svd_full(tensor)

The ranks of complete 0-th core is (1, 3, 3)
The ranks of complete 1-th core is (3, 4, 10)
The ranks of complete 2-th core is (10, 5, 2)
The ranks of complete 3-th core is (2, 2, 1)


In [4]:
ranks =  [3, 8, 2] #predefined rank

In [5]:
# TT-SVD decomposition
tt_cores_2 = tt_svd_rank(tensor, ranks)

The ranks of truncated 0-th core is (1, 3, 3)
The ranks of truncated 1-th core is (3, 4, 8)
The ranks of truncated 2-th core is (8, 5, 2)
The ranks of truncated 3-th core is (2, 2, 1)


In [6]:
tt_cores_3 = tt_svd_thres(tensor, threshold=0.5)

The ranks of truncated 0-th core is (1, 3, 3)
The ranks of truncated 1-th core is (3, 4, 7)
The ranks of truncated 2-th core is (7, 5, 2)
The ranks of truncated 3-th core is (2, 2, 1)


In [7]:


# tensor reconstruction
recon_tensor_1 = tt_svd_recon(tt_cores_1).squeeze()
recon_tensor_2 = tt_svd_recon(tt_cores_2).squeeze()
recon_tensor_3 = tt_svd_recon(tt_cores_3).squeeze()

error_comp = np.linalg.norm(tensor - recon_tensor_1)
error_trun1 = np.linalg.norm(tensor - recon_tensor_2)
error_trun2 = np.linalg.norm(tensor - recon_tensor_3)
print("The l2 norm error of complete TT-SVD is:", error_comp)
print("The l2 norm error of truncated TT-SVD (ranks truncation) is:", error_trun1)
print("The l2 norm error of truncated TT-SVD (threshold truncation) is:", error_trun2)

The l2 norm error of complete TT-SVD is: 4.9902166799147944e-14
The l2 norm error of truncated TT-SVD (ranks truncation) is: 0.3026751523226158
The l2 norm error of truncated TT-SVD (threshold truncation) is: 0.5628335461911576


### Comparison between TT-SVD and SVD

### Test 1: Reconstruct tensor using low rank tensor cores

In [8]:
cores_lr = []
cores_lr.append(np.random.rand(*(1,10,10)))
cores_lr.append(np.random.rand(*(10,10,7)))
cores_lr.append(np.random.rand(*(7,8,7)))
cores_lr.append(np.random.rand(*(7,6,1)))
test_tensor = tt_svd_recon(cores_lr).squeeze()


In [9]:
tt_cores = tt_svd_rank(tensor=test_tensor, ranks=(8, 5, 5))
# tensor reconstruction
recon_tensor = tt_svd_recon(tt_cores).squeeze()


error = np.linalg.norm(test_tensor - recon_tensor)
print("The l2 norm error is:", error)
num_elements = 0
for i in range(len(tt_cores)):
    num_elements += tt_cores[i].size
print("The numbers of elements in tt-svd is:", num_elements)

The ranks of truncated 0-th core is (1, 10, 8)
The ranks of truncated 1-th core is (8, 10, 5)
The ranks of truncated 2-th core is (5, 8, 5)
The ranks of truncated 3-th core is (5, 6, 1)
The l2 norm error is: 9.95438751112889
The numbers of elements in tt-svd is: 710


### Using similar amounts of elements for SVD

In [10]:
unfold_tensor = test_tensor.reshape(test_tensor.shape[0], -1)
print("the shape of unfolding tensor is", unfold_tensor.shape)

U, S, V = svd_full(X=unfold_tensor)
assert V.shape[1]<num_elements, "Number of elements are too few to keep the shape of V matrix. Please increase elements of TT_SVD!"
r_des = int(num_elements/(U.shape[0]+1+V.shape[1]))+1 #U.shape[0]*r + r + r*V.shape[1] = num_elements
U_new = U[:, :r_des]
S_new = S[:r_des]
V_new = V[:r_des, :]
print(f"U shape: {U_new.shape}, S shape: {S_new.shape}, V shape: {V_new.shape}")
recon_tensor = svd_recon(U_new, S_new, V_new).squeeze()
error = np.linalg.norm(unfold_tensor - recon_tensor)
print("The l2 norm error of standard SVD is:", error)
num_svd_elements = U_new.size + S_new.size + V_new.size
print("The total number of elements in svd is:", num_svd_elements)

the shape of unfolding tensor is (10, 480)
U shape: (10, 2), S shape: (2,), V shape: (2, 480)
The l2 norm error of standard SVD is: 59.358439581838965
The total number of elements in svd is: 982


### What about using complete TT-SVD? (Similar accuracy)

In [11]:
cores_lr = []
cores_lr.append(np.random.rand(*(1,10,3)))
cores_lr.append(np.random.rand(*(3,10,4)))
cores_lr.append(np.random.rand(*(4,8,2)))
cores_lr.append(np.random.rand(*(2,6,1)))
test_tensor = tt_svd_recon(cores_lr).squeeze()


In [12]:
tt_cores = tt_svd_thres(tensor=test_tensor, threshold=0.00001)
# tensor reconstruction
recon_tensor = tt_svd_recon(tt_cores).squeeze()


error = np.linalg.norm(test_tensor - recon_tensor)
print("The l2 norm error is:", error)
num_elements = 0
for i in range(len(tt_cores)):
    num_elements += tt_cores[i].size
print("The numbers of elements in tt-svd is:", num_elements)

The ranks of truncated 0-th core is (1, 10, 3)
The ranks of truncated 1-th core is (3, 10, 4)
The ranks of truncated 2-th core is (4, 8, 2)
The ranks of truncated 3-th core is (2, 6, 1)
The l2 norm error is: 2.2903358795914637e-13
The numbers of elements in tt-svd is: 226


In [13]:
unfold_tensor = test_tensor.reshape(test_tensor.shape[0]*test_tensor.shape[1], -1)
print("the shape of unfolding tensor is", unfold_tensor.shape)

U, S, V = svd_full(X=unfold_tensor)
print(f"U shape: {U.shape}, S shape: {S.shape}, V shape: {V.shape}")
recon_tensor = svd_recon(U, S, V).squeeze()
error = np.linalg.norm(unfold_tensor - recon_tensor)
print("The l2 norm error of standard SVD is:", error)
num_svd_elements = U.size + S.size + V.size
print("The total number of elements in svd is:", num_svd_elements)

the shape of unfolding tensor is (100, 48)
U shape: (100, 48), S shape: (48,), V shape: (48, 48)
The l2 norm error of standard SVD is: 9.824587744920143e-14
The total number of elements in svd is: 7152
